In [ ]:
import os
import json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix
import xgboost as xgb

Lets load the parameters we have experimented with earlier:

In [ ]:
def load_unique_hyperparameters(jsonl_file_path: str) -> list[dict]:
    """Scans a JSONL file line-by-line and extracts unique hyperparameter configurations.

    Args:
        jsonl_file_path (str): Path to the .jsonl file.

    Returns:
        list[dict]: List of unique hyperparameter dictionaries.
    """
    unique_params = []
    seen_configs = set()

    with open(jsonl_file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            record = json.loads(line)
            params = record.get("hyperparameters", {})

            # Serialize dict with sorted keys to make it hashable for set lookup
            config_key = json.dumps(params, sort_keys=True)

            if config_key not in seen_configs:
                seen_configs.add(config_key)
                unique_params.append(params)

    return unique_params

In [ ]:
eval_metric = "mglogloss" # either mlogloss or merror
objective = "multi:softprob" # multi:softmax or multi:softprob
n_splits = 5

In [ ]:
json_path = rf'../dataset/experiment_logs.jsonl'

raw_data = pd.read_csv(rf'../dataset/full.csv', index_col=False)

pruned_data = pd.read_csv(rf'../dataset/full_v5.csv', index_col=False)

pca_data = pd.read_csv(rf'../dataset/full_v5_pca.csv', index_col=False)

# Target variable, the same across all data permuatations so we just take it from the raw_data dataframe and use this on each permutation of the dataframes during training in the indices
raw_data_y = pd.DataFrame(raw_data['Activity'])

# LabelEncoder for the target variable
label_encoder = LabelEncoder()
raw_data_y_encoded = label_encoder.fit_transform(raw_data_y)

unique_groups = raw_data['subject'].unique()

groups = pd.DataFrame(raw_data['subject'])

# Parameters without duplicates, without random_state, and eval_metric
parameters = load_unique_hyperparameters(json_path)

# Mulitclass eval_metrics: mlogloss, merror
# Mutliclass objective: multi:softprob, multi:softmax

# We can safely remove the target variables and the group names:

raw_data = raw_data.drop(columns=['Activity', 'subject'])
pruned_data = pruned_data.drop(columns=['Activity', 'subject'])
pca_data = pca_data.drop(columns=['Activity', 'subject'])

In [ ]:
group_kfold = GroupKFold(n_splits=n_splits)

Going through the parameters with the raw data:

In [ ]:
# One object in JSON pr instance, per parameter configruation with the following structure:
# You may wrap it in different object types if the strucutre is ineffeicient:
# instances: instance({"parameters": {param_dictionary}, raw_data_results: [{fold_results}, {final_results}]}, pruned_data_results: [{...}], pca_data_results: [{...}]), 
# instance({ ...})
# 
# each instance represent one parameter configruation and each instance has results for each fold in the cross-validation as well as the final results
# include valuable metrics in the results such as accuracy, confusion matrix, f1-score, precision, recall, etc. for each fold and the final results
#
#
#
#
#
#

for param in parameters:

    xgb_model = None

    xgb_model = xgb.XGBClassifier(
        **param,
        random_state=69,
        eval_metric=eval_metric,
        objective=objective,
        use_label_encoder=False
    )

    print(f"Testing {str(param)} with the raw dataset")
    for fold, (train_idx, test_idx) in enumerate(group_kfold.split(raw_data, raw_data_y_encoded, groups=groups)):
        print(f"Fold {fold + 1}")
        print(f"Train indices: {train_idx}")
        print(f"Test indices: {test_idx}")
        print("-" * 40)

        # Filter the raw_data and raw_data_y_encoded based on the indices
        X_train, X_test = raw_data.iloc[train_idx], raw_data.iloc[test_idx]

        # Filter the target variable based on the indices
        y_train, y_test = raw_data_y_encoded[train_idx], raw_data_y_encoded[test_idx]

        xgb_model.fit(X_train, y_train)

    # Final resutlts for the raw data set here

    # Reset the model for the pruned dataset

    xgb_model = None

    xgb_model = xgb.XGBClassifier(
        **param,
        random_state=69,
        eval_metric=eval_metric,
        objective=objective,
        use_label_encoder=False
    )
    
    print(f"Testing {str(param)} with the pruned dataset")
    for fold, (train_idx, test_idx) in enumerate(group_kfold.split(pruned_data, raw_data_y_encoded, groups=groups)):
        print(f"Fold {fold + 1}")
        print(f"Train indices: {train_idx}")
        print(f"Test indices: {test_idx}")
        print("-" * 40)

        # Filter the pruned_data and raw_data_y_encoded based on the indices
        X_train, X_test = pruned_data.iloc[train_idx], pruned_data.iloc[test_idx]

        # Filter the target variable based on the indices
        y_train, y_test = raw_data_y_encoded[train_idx], raw_data_y_encoded[test_idx]

        xgb_model.fit(X_train, y_train)


    # Final resutlts for the raw data set here

    # Reset the model for the PCA dataset

    xgb_model = None

    xgb_model = xgb.XGBClassifier(
        **param,
        random_state=69,
        eval_metric=eval_metric,
        objective=objective,
        use_label_encoder=False
    )
    
    print(f"Testing {str(param)} with the PCA dataset")
    for fold, (train_idx, test_idx) in enumerate(group_kfold.split(pca_data, raw_data_y_encoded, groups=groups)):
        print(f"Fold {fold + 1}")
        print(f"Train indices: {train_idx}")
        print(f"Test indices: {test_idx}")
        print("-" * 40)

        # Filter the pca_data and raw_data_y_encoded based on the indices
        X_train, X_test = pca_data.iloc[train_idx], pca_data.iloc[test_idx]

        # Filter the target variable based on the indices
        y_train, y_test = raw_data_y_encoded[train_idx], raw_data_y_encoded[test_idx]

        xgb_model.fit(X_train, y_train)

In [ ]:
from datetime import datetime
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb


def load_unique_hyperparameters(jsonl_file_path: str) -> list[dict]:
    """Scans a JSONL file line-by-line and extracts unique hyperparameter configurations."""
    unique_params = []
    seen_configs = set()

    with open(jsonl_file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            record = json.loads(line)
            params = record.get("hyperparameters", {})

            # Serialize dict with sorted keys to make it hashable for set lookup
            config_key = json.dumps(params, sort_keys=True)

            if config_key not in seen_configs:
                seen_configs.add(config_key)
                unique_params.append(params)

    return unique_params


def compute_metrics(y_true, y_pred, class_names):
    """Calculates accuracy, precision, recall, macro F1, and normalized confusion matrix dict."""
    acc = float(accuracy_score(y_true, y_pred))
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    # Confusion matrix in percentage per true class
    cm = confusion_matrix(y_true, y_pred)
    with np.errstate(divide="ignore", invalid="ignore"):
        cm_pct = np.true_divide(cm, cm.sum(axis=1, keepdims=True)) * 100
        cm_pct = np.nan_to_num(cm_pct)

    cm_dict = {
        str(class_names[i]): {
            str(class_names[j]): float(round(cm_pct[i, j], 2))
            for j in range(len(class_names))
        }
        for i in range(len(class_names))
    }

    return {
        "accuracy": round(acc, 4),
        "precision_macro": round(float(precision), 4),
        "recall_macro": round(float(recall), 4),
        "f1_macro": round(float(f1), 4),
        "confusion_matrix_pct": cm_dict,
    }


def run_cv_experiment(
    X, y, groups, group_kfold, param, eval_metric, objective, class_names
):
    """Executes GroupKFold cross-validation on a dataset permutation and collects results."""
    fold_results = []
    all_y_true = []
    all_y_pred = []

    # Clean params to prevent parameter collision
    param_clean = {
        k: v
        for k, v in param.items()
        if k not in ["eval_metric", "objective", "random_state"]
    }

    for fold, (train_idx, test_idx) in enumerate(
        group_kfold.split(X, y, groups=groups)
    ):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Re-initialize a fresh model instance for every fold
        xgb_model = xgb.XGBClassifier(
            **param_clean,
            random_state=69,
            eval_metric=eval_metric,
            objective=objective,
        )
        xgb_model.fit(X_train, y_train)

        y_pred = xgb_model.predict(X_test)

        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)

        fold_metrics = compute_metrics(y_test, y_pred, class_names)
        fold_metrics["fold"] = fold + 1
        fold_results.append(fold_metrics)

    # Compute overall out-of-fold (OOF) metrics across all combined test sets
    overall_results = compute_metrics(all_y_true, all_y_pred, class_names)
    overall_results["mean_fold_accuracy"] = round(
        float(np.mean([f["accuracy"] for f in fold_results])), 4
    )
    overall_results["std_fold_accuracy"] = round(
        float(np.std([f["accuracy"] for f in fold_results])), 4
    )

    return {"fold_results": fold_results, "final_results": overall_results}


# --- Script Execution ---

eval_metric = "mlogloss"  # Fixed typo: changed 'mglogloss' -> 'mlogloss'
objective = "multi:softprob"
n_splits = 5
json_path = r"../dataset/experiment_logs.jsonl"
output_results_path = r"../dataset/xgb_permutation_results.json"

# Load datasets
raw_data = pd.read_csv(r"../dataset/full.csv", index_col=False)
pruned_data = pd.read_csv(r"../dataset/full_v5.csv", index_col=False).drop(columns=["split"])
pca_data = pd.read_csv(r"../dataset/full_v5_pca.csv", index_col=False).drop(columns=["split"])

# Extract targets and group variables as 1D arrays/Series
groups = raw_data["subject"]
label_encoder = LabelEncoder()
raw_data_y_encoded = label_encoder.fit_transform(raw_data["Activity"])

# Separate features from meta-columns
raw_data = raw_data.drop(columns=["Activity", "subject"], errors="ignore")
pruned_data = pruned_data.drop(columns=["Activity", "subject"], errors="ignore")
pca_data = pca_data.drop(columns=["Activity", "subject"], errors="ignore")

group_kfold = GroupKFold(n_splits=n_splits)
parameters = load_unique_hyperparameters(json_path)

datasets = {
    "raw_data_results": raw_data,
    "pruned_data_results": pruned_data,
    "pca_data_results": pca_data,
}

instances = []

for name, df in [("raw_data", raw_data), ("pruned_data", pruned_data), ("pca_data", pca_data)]:
    non_numeric = df.select_dtypes(exclude=['number']).columns.tolist()
    print(f"{name} non-numeric columns:", non_numeric)

# Main Loop over hyperparameter combinations
for idx, param in enumerate(parameters):
    print(
        f"\n=== Running Parameter Set {idx + 1}/{len(parameters)} ==="
    )

    instance = {
        "instance_id": idx + 1,
        "parameters": param,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    for ds_name, ds_df in datasets.items():
        print(f"  -> Testing dataset permutation: {ds_name}")
        results = run_cv_experiment(
            X=ds_df,
            y=raw_data_y_encoded,
            groups=groups,
            group_kfold=group_kfold,
            param=param,
            eval_metric=eval_metric,
            objective=objective,
            class_names=label_encoder.classes_,
            cpu_count=-1
        )
        instance[ds_name] = results

    instances.append(instance)

# Save structured dataset to JSON
output_payload = {"experiment_count": len(instances), "instances": instances}

with open(output_results_path, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, indent=4)

print(f"\nAll experiments complete! Results saved to {output_results_path}")